# Annotation

## Settings 

In [ ]:
# Path related settings (these should be the same as for the previous notebook)
output_dir = 'output/'
test = 'V6'

# Gene annotation
GTF_PATH = "reference/mus_musculus.104.genes.gtf"
ENSEMBL_DATASET = "mmusculus_gene_ensembl" # This is used for replaceing ensembl gene IDs with gene names if possible
                                           # use "hsapiens_gene_ensembl" for homo sapiens
# Cell type annotation
SPECIES = "Mm" # Currently only Hs (Homo Sapiens) and Mm (Mus Musculus) are available for custom cell type annotation
SCSA_SPECIES = None # Currently only human (Homo Sapiens) and mouse (Mus Musculus) are available for SCSA
                    # None if using a custom DB file
# Optional custom database for SCSA
SCSA_CUSTOM_DB_PATH = "reference/panglao_all_mandatory_2020"
# Path to panglao markers
PANGLAO_DB_PATH = "reference/panglao_markers"

# The clustering resolutions which will be annotated
RESOLUTIONS = ['leiden_0.1', 'leiden_0.2', 'leiden_0.3', 'leiden_0.4', 'leiden_0.5', 'leiden_0.6', 'leiden_0.7', 'leiden_0.8', 'leiden_0.9','clustering']#"leiden_0.5" # Choose "None" to keep the selected clustering resolution from the previous notebook
                           # You can also choose a column which contains the clustering information in the obs table
                           # It is also possible to add more than one column (resolutions) by using a list of columns
        
MODIFY_CELL_TYPES = False # If true you get the opportunity to manually modify the automatic annotated cell types

# Save figures
save_figs = True

## Loading packages and setup

In [ ]:
import sctoolbox.atac_tree as sub_tree
import sctoolbox.annotation as annot
import sctoolbox.custom_celltype_annotation as ctannot
from sctoolbox.celltype_annotation import run_scsa
import pandas as pd
import episcanpy as epi
import scanpy as sc
from matplotlib import pyplot as plt
from pybiomart import Dataset             #dependecy missing --> worked after installation by hand
import os
import shutil

## Check files 

In [ ]:
files = [GTF_PATH, SCSA_CUSTOM_DB_PATH, PANGLAO_DB_PATH]
for file in files:
    if os.path.isfile(file):
        print(f"{file} exists.")
    else:
        print(f"Please make sure that your input is correct. {file} does not exist.")

## Setup path handling object 

In [ ]:
# make an instance of the class
tree = sub_tree.ATAC_tree()
# set processing/output directory
tree.processing_dir = output_dir
# set sample/experiment.. 
tree.run = test

In [ ]:
tree.gtf_path = GTF_PATH

## Load anndata

In [ ]:
clustering_output = tree.clustering_anndata
adata = epi.read_h5ad(clustering_output)
adata.var

In [ ]:
clustering_output

In [ ]:
if RESOLUTIONS:
    if type(RESOLUTIONS) == str:
        RESOLUTIONS = [RESOLUTIONS]
else:
    RESOLUTIONS = adata.uns['infoprocess']['leiden_res']

## Check columns

In [ ]:
for column in RESOLUTIONS:
    if column in adata.obs:
        print(f"The obs table contains the column {column}.")
    else:
        print(f"{column} was not found in the obs table. Please pick a valid column instead.")

In [ ]:
# fix dtypes of peak coordinate columns
adata.var['peak_chr'] = adata.var['peak_chr'].astype(str)
adata.var['peak_start'] = adata.var['peak_start'].astype(int)
adata.var['peak_end'] = adata.var['peak_end'].astype(int)

In [ ]:
adata.var

## Gene annotation 

In [ ]:
annot.annotate_adata(adata,
                   GTF_PATH,
                   config=None,
                   best=True,
                   threads=6,
                   coordinate_cols=None,
                   temp_dir="",
                   verbose=True,
                   inplace=True)

In [ ]:
adata.var['gene_name']

In [ ]:
assigned_features = adata.var['gene_name'].dropna()
assigned_features

### Replace ensembl gene IDs with gene names if possible

In [ ]:
dataset = Dataset(name=ENSEMBL_DATASET,
                  host='http://www.ensembl.org')

df = dataset.query(attributes=['ensembl_gene_id', 'external_gene_name'])

In [ ]:
ens_dict = dict(zip(df['Gene stable ID'], df['Gene name']))

In [ ]:
count = 0
for index, gene in enumerate(assigned_features):
    if gene.startswith("ENSG"):
        if gene in ens_dict.keys():
            if isinstance(ens_dict[gene], str):
                assigned_features.replace(gene, ens_dict[gene], inplace=True)
                count += 1
print(f'{count} ensembl gene ids have been replaced with gene names')

In [ ]:
assigned_features

In [ ]:
assigned_adata = adata[:, assigned_features.index]
assigned_adata

## Rank peaks

In [ ]:
peaks_adata = assigned_adata.copy()

In [ ]:
#peaks_adata.uns[""]

In [ ]:
#add mario --> if noramlization is tfidf. here it runs into an error, as log1p is not given!!
#peaks_adata.uns['log1p']['base'] = None

In [ ]:
for column in RESOLUTIONS:
    print(f'Ranking peaks for clusters using obs column {column}')
    epi.tl.rank_features(peaks_adata, f'{column}', omic='ATAC', 
                         key_added=f'rank_features_groups_{column}', n_features=100, use_raw=False)
    
print('Done')

## Rank genes
Replace peaks with gene names

In [ ]:
genes_adata = peaks_adata.copy()
genes_adata.var.reset_index(inplace=True)  # remove peaks from index and save them in the column ['index']
genes_adata.var.set_index('gene_name',inplace=True)  # set genes as index
genes_adata.var.index = genes_adata.var.index.astype('str')  # to avoid index being categorical
genes_adata.var_names_make_unique(join='_')
genes_adata.raw = genes_adata

In [ ]:
for column in RESOLUTIONS:
    print(f'Ranking genes groups for clusters using obs column {column}')
    sc.tl.rank_genes_groups(genes_adata, groupby=f'{column}', use_raw=False, key_added=f'rank_genes_groups_{column}')
    
print('Done')

### Plot ranked genes

In [ ]:
for column in RESOLUTIONS:
    print(f'Plotting ranked genes groups for clustering column {column}')
    plot_key = 'rank_genes_groups_' + str(column)
    sc.pl.rank_genes_groups(genes_adata, key=plot_key, show=False)
    # TODO
    # if save_figs:
    #    plt.savefig(f'{OUTPUT_FIGS}/ranked_genes_groups_leiden_{res}.png')
    plt.show()

In [ ]:
for column in RESOLUTIONS:
    print(f'Plotting matrixplot for clustering column {column}')
    plot_key = 'rank_genes_groups_' + str(column)
    sc.pl.rank_genes_groups_matrixplot(genes_adata, standard_scale='var', n_genes=10, key=plot_key, show=False)
    # TODO
    # if save_figs:
    #    plt.savefig(f'{OUTPUT_FIGS}/ranked_genes_groups_matrixplot_leiden_{res}.png')
    plt.show()

## Write lists with ranked genes to excel

In [ ]:
#TODO
#write tables per geneset investigated, put clusters as panels into excel file


## Cell type annotation

In [ ]:
for column in RESOLUTIONS:
    ctannot.annot_ct(adata=adata, genes_adata=genes_adata, output_path=tree.annotation_dir, 
                     db_path=f"{PANGLAO_DB_PATH}", cluster_path=None, cluster_column=f"{column}", 
                     rank_genes_column=f"rank_genes_groups_{column}", sample=tree.run, 
                     ct_column=f"cell_types_{column}", tissue="all", db="panglao", inplace=True, species=SPECIES)

In [ ]:
for column in RESOLUTIONS:
    sc.pl.umap(adata, color=[f'cell_types_{column}', f'{column}'], wspace=0.5)

In [ ]:
#adata.var

In [ ]:
for column in RESOLUTIONS:
    print(f"Tables of cell type annotation with clustering {column}")
    ctannot.show_tables(annotation_dir=tree.annotation_dir, n=5, clustering_column=column)

### Modify wrong cluster annotations by selecting other recommendations

In [ ]:
if MODIFY_CELL_TYPES:
# Pick the cell type column of the obs table you want to modify as well as the corresponding clustering column
    cell_type_column = "cell_types_leiden_0.4"
    clustering_column = "leiden_0.4"

    ctannot.modify_ct(adata=adata, annotation_dir=tree.annotation_dir, clustering_column=clustering_column, 
                      cell_type_column=cell_type_column, inplace=True)

### SCSA cell type annotation

Removing underscores from gene names

In [ ]:
genes_adata.var.index = pd.Index([name.split('_')[0] for name in genes_adata.var.index])

Rank new gene names

In [ ]:
for column in RESOLUTIONS:
    print(f'Ranking genes groups for clusters using obs column {column}')
    sc.tl.rank_genes_groups(genes_adata, groupby=f'{column}', use_raw=False, key_added=f'rank_genes_groups_SCSA_{column}')

print('Done')

In [ ]:
genes_adata.uns

In [ ]:
for column in RESOLUTIONS:
    run_scsa(genes_adata,
             gene_column=None,
             gene_symbol='auto',
             key=f'rank_genes_groups_SCSA_{column}',
             column_added=f'SCSA_pred_celltype_{column}',
             inplace=True,
             python_path=None,
             scsa_path=None,
             wholedb_path=None,
             species=None,
             fc=1.5,
             pvalue=0.01,
             tissue='All',
             celltype='normal',
             user_db=SCSA_CUSTOM_DB_PATH,
             z_score='best',
             )
    adata.obs[f'SCSA_pred_celltype_{column}'] = genes_adata.obs[f'SCSA_pred_celltype_{column}']
    sc.pl.umap(adata, color=[f'SCSA_pred_celltype_{column}', f'cell_types_{column}'], wspace=0.5)


# add mario Marker annotation


In [ ]:
#Known marker genes
#arker_genes_dict = {"Cardiomyocytes": ["Mhrt","Myh7","Irx4","Myh6","Mypbc3","Tnni3","Tnnt2"],
#                    "T-cells": ["Il2ra","Cd52","Cd96","Cd8a"],
#                    "B-cells": ["Cd10","Cd27","Cd20"],
#                    "Endothelial-cells": ["Ets1","Cd34","Vcam1","Tie1","Cdh5","Nos3","Kdr","Pecam1"],
#                    "Macrophages": ["Sirpa","Itgam","Cd68","Csfr","Cd14","Cd74","Fcgr1"],
#                    "Pericytes": ["Rgs5","Kcnj8","Mcam","Pdgfrb","Notch3","Abcc9","Nes"],
#                    "EC-venous": ["Emcn","Nrp2","Flt4","Lefty2","Ephb4"],
 #                   "EC-aterial": ["Depp1","Jag2","Notch4","Notch1","Bmx"],
#                    "EC-coronary":["Pecam1","Gata4","Ets1","Fabp4"],
#                    "EC-endocardium":["Kdr","Etv2","Nrg1","Npr3"],
#                    "Skeletal-muscle":["Sgca","Gdf11","Fabp3","Ncam1","Mcam","Itga7","Tnnt1"],
#                    "Smooth Muscle cells":["Esam","Myl9","Myh11","Gata4","Cnn1","Acta2"],
#                    "Oestocytes":["Sost","Gapg"]
   #                }
marker_genes_dict = {"Cardiomyocytes": ["Mhrt","Myh7","Irx4","Myh6","Tnni3","Tnnt2"],
                     "T-cells": ["Il2ra","Cd52","Cd96","Cd8a"],
                     "B-cells": ["Cd27"],
                     "Endothelial-cells": ["Ets1","Cd34","Vcam1","Tie1","Cdh5","Nos3","Kdr","Pecam1"],
                     "Macrophages": ["Sirpa","Itgam","Cd68","Cd14","Cd74","Fcgr1"],
                     "Pericytes": ["Rgs5","Kcnj8","Mcam","Pdgfrb","Notch3","Abcc9","Nes"],
                     "EC-venous": ["Emcn","Nrp2","Flt4","Lefty2","Ephb4"],
                     "EC-aterial": ["Depp1","Jag2","Notch4","Notch1"],
                     "EC-coronary":["Pecam1","Gata4","Ets1","Fabp4"],
                     "EC-endocardium":["Kdr","Etv2","Nrg1","Npr3"],
                     "Skeletal-muscle":["Sgca","Gdf11","Fabp3","Ncam1","Mcam","Itga7","Tnnt1"],
                     "Smooth Muscle cells":["Esam","Myl9","Myh11","Gata4","Cnn1","Acta2"],
                     "Oestocytes":["Sost"]
                    }

In [ ]:
import sctoolbox.plotting as pl
import sctoolbox.utilities as utils
figure_path =output_dir


marker_gene_names = []
marker_gene_titles = []
for key, genes in marker_genes_dict.items():
    if isinstance(genes, str):
        genes = [genes]
        
    for gene in genes:
        marker_gene_names.append(gene)
        marker_gene_titles.append(f"{gene} ({key})")
    sc.pl.umap(genes_adata, color=marker_gene_names, title=marker_gene_titles, cmap=pl.sc_colormap(), ncols=3, show=False)
    utils.save_figure(figure_path + key+"_marker_gene_expression_umap.pdf")    
    marker_gene_names = []
    marker_gene_titles = []

In [ ]:
marker_gene_names

In [ ]:
#sc.pl.umap(genes_adata, color=marker_gene_names, title=marker_gene_titles, cmap=pl.sc_colormap(), ncols=3, show=False)
#utils.save_figure(figure_path + "marker_gene_expression_umap.pdf")

In [ ]:
#adata2=adata.copy()

In [ ]:
#adata2.var

## Save notebook and adata object

In [ ]:
repo_path = os.getcwd()
notebook_name = '5_annotation.ipynb'
notebook_path = os.path.join(repo_path, notebook_name)
notebook_copy = os.path.join(tree.annotation_dir , notebook_name)
shutil.copyfile(notebook_path, notebook_copy)

In [ ]:
#workaround for cellxgene error in context with latest scanpy verion
#https://github.com/chanzuckerberg/cellxgene/issues/2598#issuecomment-1377653292
import numpy as np
def rgba_hex_colors_to_rgb(adata):
    for key in adata.uns.keys():
        if key.endswith('colors'):
            adata.uns[key] = np.array([(c if len(c)<=7 else c[:-2]) for c in adata.uns[key]])

rgba_hex_colors_to_rgb(adata)

In [ ]:
#adata2.var.reset_index(inplace=True)  # remove peaks from index and save them in the column ['index']
#adata2.var.set_index('gene_name',inplace=True)  # set genes as index
#adata2.var.index = adata2.var.index.astype('str')  # to avoid index being categorical
#adata2.var_names_make_unique(join='_')
adata.var.reset_index(inplace=True)  # remove peaks from index and save them in the column ['index']
adata.var.set_index('gene_name',inplace=True)  # set genes as index
adata.var.index = adata.var.index.astype('str')  # to avoid index being categorical
adata.var_names_make_unique(join='_')

In [ ]:
adata_output = tree.annotation_anndata
#adata.write(filename=adata_output)
#adata2.write(filename=adata_output)

In [ ]:
#adata_output = "output/V6/annotation/anndata/V6_index.h5ad"
adata_output

In [ ]:
#adata2.write(filename=adata_output)

## clean up object for cellxgene


In [ ]:
for col in adata.obs:
    print(col)

In [ ]:
#to drop columns
columns=["chr1_reads","chr10_reads","chr11_reads","chr12_reads","chr13_reads","chr14_reads","chr15_reads","chr16_reads","chr17_reads","chr18_reads","chr19_reads","chr2_reads","chr3_reads","chr4_reads","chr5_reads","chr6_reads","chr7_reads","chr8_reads","chr9_reads","chrM_reads","chrX_reads","chrY_reads","id","file","Sample_new","leiden","clustering","cell_types_clustering"]

In [ ]:
adata.obs=adata.obs.drop(columns, axis=1)

In [ ]:
#adata.obs

In [ ]:
adata.write(filename=adata_output)